# Digital Forensics & Malware Analysis Lab

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


## Project overview

**Objective:** Demonstrate a repeatable first-pass investigation of suspicious artifacts without executing real malware.

**Workflow:** Collect synthetic evidence, calculate identifiers, extract static indicators, organize findings, and build an investigation summary.

**Expected output:** Artifact metadata, indicators of compromise, severity context, and a structured analyst report.

**Skills demonstrated:** DFIR, static analysis, indicators of compromise, evidence handling, Python

### Limitations

This is a training workflow. Real investigations require validated forensic acquisition, chain of custody, isolated tooling, and case-specific expertise.


In [ ]:


import subprocess
import sys
import os
import json
import hashlib
import base64
import re
import time
import random
import zipfile
import io
import struct
from datetime import datetime, timedelta
import binascii
from collections import Counter
import urllib.parse
import socket

# Install required packages
def install_dependencies():
    """Install required packages for forensics analysis"""
    packages = [
        'pycryptodome',
        'yara-python',
        'python-magic-bin',
        'requests',
        'matplotlib',
        'seaborn',
        'plotly',
        'networkx',
        'pandas'
    ]

    for package in packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
            print(f"✓ {package} installed")
        except:
            print(f"⚠ {package} installation failed")

print("🔧 Installing forensics toolkit...")
install_dependencies()

try:
    import magic
except:
    print("⚠ python-magic not available")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
import networkx as nx
from Crypto.Cipher import AES, DES
from Crypto.Random import get_random_bytes

class DigitalForensicsLab:
    def __init__(self):
        self.evidence_storage = {}
        self.case_timeline = []
        self.malware_signatures = self.load_malware_signatures()
        self.network_artifacts = []
        self.file_artifacts = []

    def display_banner(self):
        """Display the forensics lab banner"""
        banner = """
╔══════════════════════════════════════════════════════════════════════╗
║                    DIGITAL FORENSICS INVESTIGATION LAB              ║
║                         Advanced Cyber Analysis Suite               ║
║                              Colab Edition v3.0                     ║
╠══════════════════════════════════════════════════════════════════════╣
║  🔍 File Analysis  📱 Mobile Forensics  🌐 Network Investigation    ║
║  🦠 Malware Analysis  📊 Data Recovery  🔐 Cryptographic Analysis   ║
╚══════════════════════════════════════════════════════════════════════╝
        """
        print(banner)

    def load_malware_signatures(self):
        """Load malware detection signatures"""
        return {
            'ransomware': [
                b'CryptoLocker', b'WannaCry', b'Petya', b'BadRabbit',
                b'\.encrypted', b'\.locked', b'\.crypto', b'HOW_TO_DECRYPT'
            ],
            'trojan': [
                b'backdoor', b'keylogger', b'remote_access', b'rat_client',
                b'bot_command', b'C&C_server'
            ],
            'virus': [
                b'EICAR-STANDARD-ANTIVIRUS-TEST-FILE',
                b'virus_signature', b'infection_marker'
            ],
            'suspicious_strings': [
                b'CreateRemoteThread', b'VirtualAlloc', b'WriteProcessMemory',
                b'CreateProcess', b'RegSetValue', b'ShellExecute'
            ]
        }

    def create_sample_evidence(self):
        """Generate sample evidence files for analysis"""
        print("🧪 Creating sample evidence files...")

        # Create suspicious executable sample
        suspicious_exe = b"MZ\x90\x00" + b"A" * 100  # PE header start
        suspicious_exe += b"CreateRemoteThread" + b"VirtualAlloc"
        suspicious_exe += b"C:\\Windows\\System32\\cmd.exe"
        suspicious_exe += b"powershell.exe -enc " + base64.b64encode(b"malicious command")

        # Create fake ransomware note
        ransom_note = """
        YOUR FILES ARE ENCRYPTED!

        All your important files have been encrypted with strong cryptography.
        To decrypt your files, you need to pay 0.5 BTC to this address:
        1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa

        Contact: decrypt@evil.com
        Deadline: 72 hours
        """

        # Create network log sample
        network_log = """
        2024-06-18 10:30:15 TCP 192.168.1.100:1337 -> 203.0.113.5:443 [SYN]
        2024-06-18 10:30:16 TCP 203.0.113.5:443 -> 192.168.1.100:1337 [SYN,ACK]
        2024-06-18 10:30:17 HTTP GET /download/malware.exe
        2024-06-18 10:30:18 DNS query evil-c2-server.com
        2024-06-18 10:30:19 TCP 192.168.1.100:4444 -> 185.220.101.5:9999 [DATA]
        """

        # Create email sample
        email_sample = """
        Subject: URGENT: Your account will be suspended
        From: security@bank-fake.com
        To: victim@company.com

        Dear Customer,

        We have detected suspicious activity on your account.
        Please click here to verify: http://phishing-site.evil/login

        Attachment: statement.pdf.exe (2.3MB)
        """

        # Store evidence
        self.evidence_storage = {
            'suspicious_executable': suspicious_exe,
            'ransom_note': ransom_note.encode(),
            'network_logs': network_log.encode(),
            'phishing_email': email_sample.encode(),
            'timestamps': {
                'creation_time': datetime.now() - timedelta(hours=2),
                'modification_time': datetime.now() - timedelta(hours=1),
                'access_time': datetime.now()
            }
        }

        print("✅ Sample evidence created successfully!")
        return self.evidence_storage

    def file_analysis(self, file_data, filename):
        """Comprehensive file analysis"""
        print(f"\n🔍 Analyzing file: {filename}")
        print("=" * 50)

        analysis_results = {
            'filename': filename,
            'size': len(file_data),
            'md5': hashlib.md5(file_data).hexdigest(),
            'sha1': hashlib.sha1(file_data).hexdigest(),
            'sha256': hashlib.sha256(file_data).hexdigest(),
            'entropy': self.calculate_entropy(file_data),
            'file_type': self.detect_file_type(file_data),
            'strings': self.extract_strings(file_data),
            'malware_indicators': self.scan_malware_signatures(file_data),
            'suspicious_api_calls': self.find_api_calls(file_data),
            'embedded_urls': self.extract_urls(file_data),
            'encryption_indicators': self.check_encryption(file_data)
        }

        self.display_file_analysis(analysis_results)
        return analysis_results

    def calculate_entropy(self, data):
        """Calculate Shannon entropy of data"""
        if not data:
            return 0

        entropy = 0
        for x in range(256):
            p_x = float(data.count(bytes([x]))) / len(data)
            if p_x > 0:
                entropy += - p_x * (p_x.bit_length() - 1)

        return round(entropy, 3)

    def detect_file_type(self, data):
        """Detect file type based on magic bytes"""
        magic_signatures = {
            b'\x4d\x5a': 'PE Executable',
            b'\x7f\x45\x4c\x46': 'ELF Executable',
            b'\x50\x4b\x03\x04': 'ZIP Archive',
            b'\x52\x61\x72\x21': 'RAR Archive',
            b'\x89\x50\x4e\x47': 'PNG Image',
            b'\xff\xd8\xff': 'JPEG Image',
            b'\x25\x50\x44\x46': 'PDF Document',
            b'\xd0\xcf\x11\xe0': 'Microsoft Office'
        }

        for signature, file_type in magic_signatures.items():
            if data.startswith(signature):
                return file_type

        return 'Unknown'

    def extract_strings(self, data, min_length=4):
        """Extract printable strings from binary data"""
        strings = []
        current_string = ""

        for byte in data:
            if 32 <= byte <= 126:  # Printable ASCII
                current_string += chr(byte)
            else:
                if len(current_string) >= min_length:
                    strings.append(current_string)
                current_string = ""

        if len(current_string) >= min_length:
            strings.append(current_string)

        return strings[:20]  # Return first 20 strings

    def scan_malware_signatures(self, data):
        """Scan for malware signatures"""
        indicators = []

        for category, signatures in self.malware_signatures.items():
            for signature in signatures:
                if signature in data:
                    indicators.append({
                        'category': category,
                        'signature': signature.decode('utf-8', errors='ignore'),
                        'severity': 'HIGH' if category in ['ransomware', 'trojan'] else 'MEDIUM'
                    })

        return indicators

    def find_api_calls(self, data):
        """Find suspicious Windows API calls"""
        api_calls = [
            'CreateRemoteThread', 'VirtualAlloc', 'WriteProcessMemory',
            'CreateProcess', 'RegSetValue', 'ShellExecute', 'URLDownloadToFile',
            'WinExec', 'SetWindowsHookEx', 'FindWindow'
        ]

        found_apis = []
        data_str = data.decode('utf-8', errors='ignore')

        for api in api_calls:
            if api in data_str:
                found_apis.append(api)

        return found_apis

    def extract_urls(self, data):
        """Extract URLs from file data"""
        url_pattern = re.compile(
            rb'https?://(?:[-\w.])+(?:[:\d]+)?(?:/(?:[\w/_.])*(?:\?(?:[\w&=%.])*)?(?:#(?:[\w.])*)?)?'
        )

        urls = []
        for match in url_pattern.finditer(data):
            urls.append(match.group().decode('utf-8', errors='ignore'))

        return urls[:10]  # Return first 10 URLs

    def check_encryption(self, data):
        """Check for encryption/obfuscation indicators"""
        indicators = []

        # High entropy indicates possible encryption/packing
        entropy = self.calculate_entropy(data)
        if entropy > 7.5:
            indicators.append("High entropy detected - possible encryption/packing")

        # Check for base64 patterns
        base64_pattern = re.compile(rb'[A-Za-z0-9+/]{20,}={0,2}')
        if base64_pattern.search(data):
            indicators.append("Base64 encoded data detected")

        # Check for hex patterns
        hex_pattern = re.compile(rb'[0-9A-Fa-f]{32,}')
        if hex_pattern.search(data):
            indicators.append("Hexadecimal data patterns detected")

        return indicators

    def display_file_analysis(self, results):
        """Display file analysis results"""
        print(f"📄 File: {results['filename']}")
        print(f"📏 Size: {results['size']:,} bytes")
        print(f"🔢 MD5: {results['md5']}")
        print(f"🔢 SHA1: {results['sha1']}")
        print(f"🔢 SHA256: {results['sha256']}")
        print(f"📊 Entropy: {results['entropy']} (0-8 scale)")
        print(f"📋 File Type: {results['file_type']}")

        if results['malware_indicators']:
            print(f"\n⚠️  MALWARE INDICATORS FOUND:")
            for indicator in results['malware_indicators']:
                print(f"  [{indicator['severity']}] {indicator['category']}: {indicator['signature']}")

        if results['suspicious_api_calls']:
            print(f"\n🚨 Suspicious API Calls:")
            for api in results['suspicious_api_calls']:
                print(f"  • {api}")

        if results['embedded_urls']:
            print(f"\n🌐 Embedded URLs:")
            for url in results['embedded_urls']:
                print(f"  • {url}")

        if results['encryption_indicators']:
            print(f"\n🔐 Encryption Indicators:")
            for indicator in results['encryption_indicators']:
                print(f"  • {indicator}")

        if results['strings']:
            print(f"\n📝 Extracted Strings (sample):")
            for string in results['strings'][:5]:
                print(f"  • {string[:50]}...")

    def network_forensics(self, log_data):
        """Analyze network logs for suspicious activity"""
        print("\n🌐 Network Forensics Analysis")
        print("=" * 40)

        log_str = log_data.decode('utf-8', errors='ignore')
        lines = log_str.strip().split('\n')

        connections = []
        dns_queries = []
        suspicious_ips = []

        # Parse network logs
        for line in lines:
            if 'TCP' in line:
                # Extract IP addresses and ports
                ip_pattern = r'(\d+\.\d+\.\d+\.\d+):(\d+)'
                matches = re.findall(ip_pattern, line)
                if len(matches) >= 2:
                    src_ip, src_port = matches[0]
                    dst_ip, dst_port = matches[1]
                    connections.append({
                        'src_ip': src_ip,
                        'src_port': src_port,
                        'dst_ip': dst_ip,
                        'dst_port': dst_port,
                        'timestamp': datetime.now()
                    })

            elif 'DNS' in line:
                domain_match = re.search(r'DNS query (\S+)', line)
                if domain_match:
                    dns_queries.append(domain_match.group(1))

        # Analyze for suspicious activity
        analysis = {
            'total_connections': len(connections),
            'unique_destinations': len(set(conn['dst_ip'] for conn in connections)),
            'dns_queries': dns_queries,
            'suspicious_domains': self.check_suspicious_domains(dns_queries),
            'port_analysis': self.analyze_ports(connections),
            'geographic_analysis': self.analyze_ip_geography(connections)
        }

        self.display_network_analysis(analysis)
        return analysis

    def check_suspicious_domains(self, domains):
        """Check for suspicious domain patterns"""
        suspicious = []

        suspicious_patterns = [
            r'.*-c2-.*',  # Command and control
            r'.*evil.*',  # Obviously malicious
            r'.*temp.*\.tk$',  # Temporary domains
            r'.*\d{8,}.*',  # Long number sequences
            r'.*[a-z]{20,}.*'  # Very long random strings
        ]

        for domain in domains:
            for pattern in suspicious_patterns:
                if re.match(pattern, domain, re.IGNORECASE):
                    suspicious.append(domain)
                    break

        return suspicious

    def analyze_ports(self, connections):
        """Analyze port usage patterns"""
        port_counter = Counter()

        for conn in connections:
            port_counter[conn['dst_port']] += 1

        common_malicious_ports = {
            '1337': 'Common backdoor port',
            '4444': 'Metasploit default',
            '6666': 'IRC bot/backdoor',
            '9999': 'Remote access trojan'
        }

        suspicious_ports = {}
        for port, count in port_counter.items():
            if port in common_malicious_ports:
                suspicious_ports[port] = {
                    'count': count,
                    'description': common_malicious_ports[port]
                }

        return {
            'most_common': port_counter.most_common(5),
            'suspicious_ports': suspicious_ports
        }

    def analyze_ip_geography(self, connections):
        """Analyze IP addresses for geographic indicators"""
        # Simplified geographic analysis based on IP ranges
        geographic_indicators = []

        for conn in connections:
            ip = conn['dst_ip']

            # Check for known suspicious IP ranges (simplified)
            if ip.startswith('185.220.'):  # Tor exit nodes
                geographic_indicators.append(f"{ip} - Possible Tor exit node")
            elif ip.startswith('203.0.113.'):  # RFC 5737 test range
                geographic_indicators.append(f"{ip} - Test/documentation IP")

        return geographic_indicators

    def display_network_analysis(self, analysis):
        """Display network analysis results"""
        print(f"📊 Total Connections: {analysis['total_connections']}")
        print(f"🌍 Unique Destinations: {analysis['unique_destinations']}")

        if analysis['dns_queries']:
            print(f"\n🔍 DNS Queries:")
            for domain in analysis['dns_queries']:
                print(f"  • {domain}")

        if analysis['suspicious_domains']:
            print(f"\n⚠️  SUSPICIOUS DOMAINS:")
            for domain in analysis['suspicious_domains']:
                print(f"  • {domain}")

        if analysis['port_analysis']['suspicious_ports']:
            print(f"\n🚨 SUSPICIOUS PORTS:")
            for port, info in analysis['port_analysis']['suspicious_ports'].items():
                print(f"  • Port {port}: {info['description']} ({info['count']} connections)")

        if analysis['geographic_analysis']:
            print(f"\n🌐 Geographic Indicators:")
            for indicator in analysis['geographic_analysis']:
                print(f"  • {indicator}")

    def timeline_analysis(self):
        """Create timeline of events"""
        print("\n⏰ Timeline Analysis")
        print("=" * 30)

        events = [
            {'time': datetime.now() - timedelta(hours=3), 'event': 'Suspicious email received', 'severity': 'MEDIUM'},
            {'time': datetime.now() - timedelta(hours=2), 'event': 'Executable file created', 'severity': 'HIGH'},
            {'time': datetime.now() - timedelta(hours=1.5), 'event': 'Network connection to C2 server', 'severity': 'HIGH'},
            {'time': datetime.now() - timedelta(hours=1), 'event': 'Registry modifications detected', 'severity': 'HIGH'},
            {'time': datetime.now() - timedelta(minutes=30), 'event': 'File encryption activity', 'severity': 'CRITICAL'},
            {'time': datetime.now() - timedelta(minutes=15), 'event': 'Ransom note created', 'severity': 'CRITICAL'}
        ]

        for event in sorted(events, key=lambda x: x['time']):
            time_str = event['time'].strftime('%H:%M:%S')
            severity_symbol = {
                'LOW': '🟢',
                'MEDIUM': '🟡',
                'HIGH': '🟠',
                'CRITICAL': '🔴'
            }.get(event['severity'], '⚪')

            print(f"{time_str} {severity_symbol} [{event['severity']}] {event['event']}")

        return events

    def memory_analysis(self):
        """Simulate memory dump analysis"""
        print("\n🧠 Memory Dump Analysis")
        print("=" * 35)

        # Simulate memory artifacts
        processes = [
            {'pid': 1234, 'name': 'explorer.exe', 'suspicious': False},
            {'pid': 5678, 'name': 'svchost.exe', 'suspicious': False},
            {'pid': 9999, 'name': 'backdoor.exe', 'suspicious': True},
            {'pid': 8888, 'name': 'cmd.exe', 'suspicious': True, 'parent': 'backdoor.exe'}
        ]

        print("🔍 Running Processes:")
        for proc in processes:
            status = "⚠️ SUSPICIOUS" if proc['suspicious'] else "✅ Normal"
            parent_info = f" (Parent: {proc.get('parent', 'N/A')})" if 'parent' in proc else ""
            print(f"  PID {proc['pid']}: {proc['name']} {status}{parent_info}")

        # Network connections
        print(f"\n🌐 Network Connections:")
        connections = [
            {'process': 'backdoor.exe', 'remote': '185.220.101.5:9999', 'status': 'ESTABLISHED'},
            {'process': 'explorer.exe', 'remote': '8.8.8.8:53', 'status': 'TIME_WAIT'}
        ]

        for conn in connections:
            print(f"  {conn['process']} -> {conn['remote']} [{conn['status']}]")

        # Registry analysis
        print(f"\n📝 Registry Modifications:")
        registry_changes = [
            'HKLM\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Run\\backdoor',
            'HKCU\\Software\\Classes\\exefile\\shell\\open\\command'
        ]

        for reg_key in registry_changes:
            print(f"  ⚠️  Modified: {reg_key}")

    def crypto_analysis(self):
        """Cryptographic analysis tools"""
        print("\n🔐 Cryptographic Analysis")
        print("=" * 35)

        # Sample encrypted data
        sample_data = "VGhpcyBpcyBhIHNlY3JldCBtZXNzYWdlIQ=="

        print(f"📄 Sample encrypted data: {sample_data}")

        # Try different decoding methods
        print(f"\n🔍 Decoding attempts:")

        # Base64
        try:
            decoded = base64.b64decode(sample_data).decode('utf-8')
            print(f"  ✅ Base64: {decoded}")
        except:
            print(f"  ❌ Base64: Failed")

        # ROT13
        rot13_decoded = ''.join([chr((ord(c) - ord('a') + 13) % 26 + ord('a')) if c.islower()
                                else chr((ord(c) - ord('A') + 13) % 26 + ord('A')) if c.isupper()
                                else c for c in sample_data])
        print(f"  🔄 ROT13: {rot13_decoded}")

        # Caesar cipher analysis
        print(f"\n📊 Caesar Cipher Analysis:")
        for shift in range(1, 6):
            shifted = ''.join([chr((ord(c) - ord('A') + shift) % 26 + ord('A')) if c.isupper()
                              else chr((ord(c) - ord('a') + shift) % 26 + ord('a')) if c.islower()
                              else c for c in "HELLO"])
            print(f"  Shift {shift}: HELLO -> {shifted}")

    def generate_forensics_report(self):
        """Generate comprehensive forensics report"""
        report = f"""
╔══════════════════════════════════════════════════════════════════════╗
║                         DIGITAL FORENSICS REPORT                    ║
║                           Case #DF-2024-001                         ║
║                     Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}                     ║
╚══════════════════════════════════════════════════════════════════════╝

EXECUTIVE SUMMARY:
└─ Investigation Target: Suspected malware infection and data breach
└─ Evidence Sources: Network logs, memory dumps, file system artifacts
└─ Investigation Status: CRITICAL - Active threat detected

KEY FINDINGS:
└─ ⚠️  Ransomware signatures identified in executable files
└─ ⚠️  Command & Control server communication detected
└─ ⚠️  Suspicious network traffic to known malicious IPs
└─ ⚠️  Registry modifications indicating persistence mechanism
└─ ⚠️  File encryption activity observed

TECHNICAL ANALYSIS:
└─ Malware Family: Suspected ransomware variant
└─ Attack Vector: Phishing email with malicious attachment
└─ Persistence: Registry Run key modification
└─ Data Exfiltration: Confirmed network connections to C2 servers
└─ Impact: Multiple files encrypted, ransom note deployed

TIMELINE OF EVENTS:
└─ T-03:00 - Phishing email received and opened
└─ T-02:00 - Malicious executable launched
└─ T-01:30 - C2 server communication established
└─ T-01:00 - Registry persistence installed
└─ T-00:30 - File encryption process initiated
└─ T-00:15 - Ransom note created and displayed

INDICATORS OF COMPROMISE (IOCs):
└─ File Hashes:
    • MD5: a1b2c3d4e5f6789012345678901234ab
    • SHA256: 1a2b3c4d5e6f7890abcdef1234567890abcdef1234567890abcdef1234567890
└─ Network Indicators:
    • C2 Server: 185.220.101.5:9999
    • Domain: evil-c2-server.com
└─ File System Indicators:
    • %APPDATA%\\backdoor.exe
    • Ransom note: HOW_TO_DECRYPT.txt

RECOMMENDATIONS:
└─ Immediate Actions:
    • Isolate affected systems from network
    • Preserve system state for further analysis
    • Check for lateral movement to other systems
    • Restore from clean backups if available
└─ Long-term Actions:
    • Implement advanced email filtering
    • Deploy endpoint detection and response (EDR)
    • Conduct security awareness training
    • Regular backup verification and testing

ANALYST: Digital Forensics AI Lab
REPORT VERSION: 1.0
CLASSIFICATION: CONFIDENTIAL
"""
        return report

def main():
    """Main forensics lab interface"""
    lab = DigitalForensicsLab()
    lab.display_banner()

    while True:
        print("\n" + "="*70)
        print("DIGITAL FORENSICS LAB - INVESTIGATION MENU")
        print("="*70)
        print("1. 🧪 Create Sample Evidence")
        print("2. 🔍 File Analysis & Malware Detection")
        print("3. 🌐 Network Forensics Analysis")
        print("4. ⏰ Timeline Analysis")
        print("5. 🧠 Memory Dump Analysis")
        print("6. 🔐 Cryptographic Analysis")
        print("7. 📊 Generate Forensics Report")
        print("8. 🎯 Full Investigation Workflow")
        print("9. ❌ Exit Lab")

        choice = input("\nSelect investigation tool (1-9): ").strip()

        if choice == '1':
            evidence = lab.create_sample_evidence()
            print(f"✅ Created {len(evidence)} evidence artifacts")

        elif choice == '2':
            if not lab.evidence_storage:
                print("⚠️  No evidence loaded. Creating sample evidence first...")
                lab.create_sample_evidence()

            print("\nAvailable evidence files:")
            for i, filename in enumerate(lab.evidence_storage.keys(), 1):
                if filename != 'timestamps':
                    print(f"  {i}. {filename}")

            try:
                file_choice = input("Select file number to analyze: ").strip()
                filenames = [k for k in lab.evidence_storage.keys() if k != 'timestamps']
                selected_file = filenames[int(file_choice) - 1]
                file_data = lab.evidence_storage[selected_file]
                lab.file_analysis(file_data, selected_file)
            except (ValueError, IndexError):
                print("❌ Invalid selection")

        elif choice == '3':
            if 'network_logs' in lab.evidence_storage:
                lab.network_forensics(lab.evidence_storage['network_logs'])
            else:
                print("⚠️  No network logs available. Creating sample evidence...")
                lab.create_sample_evidence()
                lab.network_forensics(lab.evidence_storage['network_logs'])

        elif choice == '4':
            lab.timeline_analysis()

        elif choice == '5':
            lab.memory_analysis()

        elif choice == '6':
            lab.crypto_analysis()

        elif choice == '7':
            report = lab.generate_forensics_report()
            print(report)

        elif choice == '8':
            print("\n🚀 Starting Full Investigation Workflow...")
            print("This will run all analysis modules in sequence.\n")

            # Create evidence if not exists
            if not lab.evidence_storage:
                lab.create_sample_evidence()

            # Run all analysis modules
            print("Phase 1: File Analysis")
            for filename, data in lab.evidence_storage.items():
                if filename != 'timestamps' and isinstance(data, bytes):
                    lab.file_analysis(data, filename)
                    print("\n" + "-"*50 + "\n")

            print("Phase 2: Network Analysis")
            if 'network_logs' in lab.evidence_storage:
                lab.network_forensics(lab.evidence_storage['network_logs'])
                print("\n" + "-"*50 + "\n")

            print("Phase 3: Timeline Analysis")
            lab.timeline_analysis()
            print("\n" + "-"*50 + "\n")

            print("Phase 4: Memory Analysis")
            lab.memory_analysis()
            print("\n" + "-"*50 + "\n")

            print("Phase 5: Cryptographic Analysis")
            lab.crypto_analysis()
            print("\n" + "-"*50 + "\n")

            print("Phase 6: Final Report Generation")
            report = lab.generate_forensics_report()
            print(report)

        elif choice == '9':
            print("\n🔬 Forensics investigation complete!")
            print("Thank you for using the Digital Forensics Lab!")
            break

        else:
            print("❌ Invalid choice. Please select 1-9.")

if __name__ == "__main__":
    main()